In [ ]:
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from app.ml_models.pricing_transformers import (
    FilteringFeatureEngineering,
    MakeCanonicalizer,
    ModelCanonicalizer,
    ModelTargetEncoder,
)

pd.options.display.max_columns = 100
pd.options.display.max_rows = 100

MODEL_INPUT_COLUMNS = [
    "make",
    "model",
    "year",
    "mileage",
    "body_type",
    "transmission",
    "fuel_type",
    "drivetrain",
    "engine_cylinders",
    "engine_volume",
    "color",
]
DATA_PATH = os.environ.get("PRICE_TRAINING_DATA", "app/ml_models/data/app_ready.csv")
MODEL_OUTPUT_PATH = os.environ.get("PRICE_MODEL_OUTPUT", "app/ml_models/car_price_pipeline.pkl")

df = pd.read_csv(DATA_PATH)
df = df[["price", *MODEL_INPUT_COLUMNS]].copy()
df = df.dropna(subset=["price", "make", "model", "year"])
df = df[df["price"] > 0].copy()


In [100]:
df.head()

,price,make,model,year,body_type,fuel_type,engine_volume,mileage,engine_cylinders,transmission,drivetrain,color
0,3607,Ford,Escape,2011.0,SUV,Hybrid,2.5,104991.0,4.0,Automatic,AWD,White
1,39493,Hyundai,Santa Fe,2016.0,SUV,Diesel,2.0,99998.0,4.0,Automatic,FWD,White
2,59464,Hyundai,Santa Fe,2016.0,SUV,Diesel,2.0,47225.0,4.0,Automatic,FWD,White
3,549,Toyota,C-HR,2018.0,SUV,Petrol,2.0,46073.0,4.0,Automatic,FWD,White
4,7683,Hyundai,Elantra,2016.0,Sedan,Petrol,1.8,75708.0,4.0,Automatic,FWD,Blue


Transformer classes live in `app.ml_models.pricing_transformers` so saved models can be loaded by the API.


Using importable transformer classes from `app.ml_models.pricing_transformers`.


Using importable transformer classes from `app.ml_models.pricing_transformers`.


Using importable transformer classes from `app.ml_models.pricing_transformers`.


In [105]:
categorical_features = ['make', 'color', 'body_type', 'fuel_type', 'drivetrain', 'transmission']
numeric_features = ['model', 'year', 'engine_volume', 'mileage', 'engine_cylinders']

numeric_pipeline = Pipeline([
    ("FeatureEngin", FilteringFeatureEngineering()),
    ("ModelCanonicalizer", ModelCanonicalizer()),
    ("ModelTargetEncoder", ModelTargetEncoder())
])

categorical_pipeline = Pipeline([
    ("MakeCanonicalizer", MakeCanonicalizer()),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(n_estimators=1000, random_state=42, n_jobs=-1))
])

In [106]:
df.head()

,price,make,model,year,body_type,fuel_type,engine_volume,mileage,engine_cylinders,transmission,drivetrain,color
0,3607,Ford,Escape,2011.0,SUV,Hybrid,2.5,104991.0,4.0,Automatic,AWD,White
1,39493,Hyundai,Santa Fe,2016.0,SUV,Diesel,2.0,99998.0,4.0,Automatic,FWD,White
2,59464,Hyundai,Santa Fe,2016.0,SUV,Diesel,2.0,47225.0,4.0,Automatic,FWD,White
3,549,Toyota,C-HR,2018.0,SUV,Petrol,2.0,46073.0,4.0,Automatic,FWD,White
4,7683,Hyundai,Elantra,2016.0,Sedan,Petrol,1.8,75708.0,4.0,Automatic,FWD,Blue


In [107]:
# -----------------------------
# Train/test split
# -----------------------------
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

X_train = train_df.drop(columns=["price"])
y_train = train_df["price"]

X_test = test_df.drop(columns=["price"])
y_test = test_df["price"]


# -----------------------------
# Fit model
# -----------------------------
model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MAE: 4421.871008519959
R2: 0.718910830603944


In [ ]:
joblib.dump(model_pipeline, MODEL_OUTPUT_PATH)
print(f"Saved model to {MODEL_OUTPUT_PATH}")
